In [1]:
import cv2
import numpy as np
import os


In [3]:

# --- Config ---
video_path = r"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\I. INTRODUCTION TO SAP HANA MM.mp4"
output_folder = "extracted_documents"
min_area = 50000  # Minimum contour area to be considered a document
similarity_threshold = 0.9  # To avoid saving near-identical frames


In [5]:

# Create folder if not exists
from regex import F


os.makedirs(output_folder, exist_ok=True)

# Function to compare images for similarity
def are_images_similar(img1, img2):
    if img1 is None or img2 is None:
        return False
    img1 = cv2.resize(img1, (200, 200))
    img2 = cv2.resize(img2, (200, 200))
    diff = cv2.absdiff(img1, img2)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    non_zero = np.count_nonzero(gray)
    total_pixels = gray.size
    similarity = 1 - (non_zero / total_pixels)
    return similarity > similarity_threshold

# Read the video
cap = cv2.VideoCapture(video_path)
frame_count = 0
saved_count = 0
last_saved = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 75, 200)

    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    for contour in contours:
        if cv2.contourArea(contour) > min_area:
            peri = cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, 0.02 * peri, True)

            if len(approx) == 4:  # Looks like a rectangle
                x, y, w, h = cv2.boundingRect(approx)
                document = frame[y:y+h, x:x+w]

                if not are_images_similar(document, last_saved):
                    filename = os.path.join(output_folder, f"doc_{saved_count}.jpg")
                    cv2.imwrite(filename, document)
                    last_saved = document.copy()
                    saved_count += 1
                break  # Only take the first (largest) rectangle

    frame_count += 1
    print(F"Processing frame {frame_count}, saved {saved_count} documents", end='\r')

cap.release()
print(f"Extraction complete. {saved_count} documents saved in '{output_folder}' folder.")


KeyboardInterrupt: 